# Lab 6 — Excel Group Checkpoint

**Day 01 · Data Science Introduction · Cisco AI/ML Training**

---

## Learning objectives

1. Build a **regional summary** (pivot / `SUMIF` logic) of Q1 and Q2 sales.
2. Count teams where **Q2 > Q1** and find the top Q2 region.
3. Compare Excel answers to the verification script.
4. Present one **bar chart** of regional Q2 totals.

> **Checkpoints:** **15** growth teams · top region **North** · total q2_sales **3006**

**Companion script:** `../scripts/lab06_excel_group_checkpoint.py`

## Excel workflow (mirror in this notebook)

1. Import `team_sales.csv` → PivotTable: rows = `region`, values = Sum of `q1_sales`, `q2_sales`.
2. Count row-level growth: `q2_sales > q1_sales`.
3. Find region with highest Q2 total.
4. Insert a column chart of regional Q2 sales.

---

## 1. Load team sales

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-01":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "hands-on" / "day-01" / "data" / "team_sales.csv").is_file():
            GH_ROOT = parent
            break

TEAM_SALES_CSV = GH_ROOT / "hands-on" / "day-01" / "data" / "team_sales.csv"
df = pd.read_csv(TEAM_SALES_CSV)

print(f"teams: {len(df)}")
print(f"regions: {df['region'].nunique()}")
display(df.head())

---

## 2. Regional totals (pivot equivalent)

In [ ]:
by_region = (
    df.groupby("region")[["q1_sales", "q2_sales"]]
    .sum()
    .assign(growth=lambda x: x["q2_sales"] - x["q1_sales"])
    .round(0)
    .astype(int)
)

print("regional totals:")
display(by_region)

---

## 3. Growth count and top region

In [ ]:
teams_with_growth = int((df["q2_sales"] > df["q1_sales"]).sum())
total_q2 = int(df["q2_sales"].sum())
top_region = by_region["q2_sales"].idxmax()

print(f"total q2_sales: {total_q2}")
print(f"teams with q2 > q1: {teams_with_growth}")
print(f"top region by q2 total: {top_region}")

---

## 4. Chart — regional Q2 sales

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
regions = by_region.index.tolist()
q2_totals = by_region["q2_sales"].values
colors = ["#c41230" if r == top_region else "#049fd9" for r in regions]

ax.bar(regions, q2_totals, color=colors, edgecolor="white")
ax.set_xlabel("region")
ax.set_ylabel("total q2_sales")
ax.set_title("Regional Q2 sales (group checkpoint chart)")
for i, val in enumerate(q2_totals):
    ax.text(i, val + 8, str(val), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

Compare this chart to the column chart your group builds in Excel — labels and ranking should match.

---

## 5. Checkpoint

In [ ]:
expected = {
    "East": (688, 757, 69),
    "North": (753, 778, 25),
    "South": (698, 722, 24),
    "West": (686, 749, 63),
}

assert len(df) == 20
assert df["region"].nunique() == 4
assert total_q2 == 3006
assert teams_with_growth == 15
assert top_region == "North"
for region, (q1, q2, growth) in expected.items():
    row = by_region.loc[region]
    assert int(row["q1_sales"]) == q1
    assert int(row["q2_sales"]) == q2
    assert int(row["growth"]) == growth
print("✓ Excel group checkpoint passed")

---

## Reflection questions

1. Why is **North** top by Q2 total but not by growth (Q2 − Q1)?
2. Which CRISP-DM phases did this lab cover without ML?
3. What changes on Day 2 when we switch from Excel to **Pandas + Zomato**?

**Previous:** [Lab 5 — Tool landscape](lab05_tool_landscape.ipynb)  
**Next:** [Day 02 — Python for Data Science](../../day-02/labs.md)